In [ ]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import os
import pandas as

In [ ]:
def descargar_pdfs_23f():
    url_base = "https://www.lamoncloa.gob.es/consejodeministros/paginas/desclasificacion-documentos-23F.aspx"
    carpeta_destino = "documentos_23F"
    
    # Crear la carpeta si no existe
    if not os.path.exists(carpeta_destino):
        os.makedirs(carpeta_destino)
        print(f"Carpeta '{carpeta_destino}' creada.")

    # Definir un User-Agent para evitar bloqueos básicos
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }

    print(f"Accediendo a la web...")
    try:
        respuesta = requests.get(url_base, headers=headers)
        respuesta.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Error al acceder a la web: {e}")
        return

    # Parsear el contenido HTML
    soup = BeautifulSoup(respuesta.text, 'html.parser')
    
    # Buscar todos los enlaces (etiquetas <a>)
    enlaces = soup.find_all('a', href=True)
    
    descargas_exitosas = 0
    
    for enlace in enlaces:
        href = enlace['href']
        
        # Filtrar solo los archivos PDF
        if href.lower().endswith('.pdf'):
            # Construir la URL completa (maneja enlaces relativos)
            url_pdf = urljoin(url_base, href)
            
            # Limpiar el nombre del archivo para guardarlo
            nombre_archivo = os.path.basename(href).split('?')[0]
            ruta_guardado = os.path.join(carpeta_destino, nombre_archivo)
            
            print(f"Descargando: {nombre_archivo}...")
            
            try:
                r_pdf = requests.get(url_pdf, headers=headers, stream=True)
                r_pdf.raise_for_status()
                
                with open(ruta_guardado, 'wb') as f:
                    for chunk in r_pdf.iter_content(chunk_size=8192):
                        f.write(chunk)
                
                descargas_exitosas += 1
            except Exception as e:
                print(f"No se pudo descargar {nombre_archivo}: {e}")

    print(f"\nProceso finalizado. Se han descargado {descargas_exitosas} archivos en la carpeta '{carpeta_destino}'.")

if __name__ == "__main__":
    descargar_pdfs_23f()